# Feature engineering — Billings
This notebook creates simple, easy-to-follow features from the processed billings file. The code is split into small cells so a beginner can follow along.

In [12]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [13]:
df = pd.read_csv("../../data/02_processed/processed_billings.csv")
df.head()

/var/folders/6q/3_ys7c717hz_scqx9xw7q35w0000gn/T/ipykernel_7136/678099606.py:1: DtypeWarning: Columns (14,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/02_processed/processed_billings.csv")


,co_ref,renewal_month,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,...,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,renewal_year,datetime_out,registration_date_missing_flag,prospect_renewal_date_missing_flag,proforma_date_missing_flag
0,VT6174,2024-01-11,8.0,42.5,799.0,9,9,7.5,9.0,True,...,2023-01-11 00:00:00,Band B,664.0,1.0,1,2024,2024-01-11,0,0,0
1,VD3828,2025-01-08,8.0,41.5,799.0,9,9,7.5,8.0,True,...,No_History,No_History,0.0,0.0,1,2025,2025-01-08,0,0,0
2,DV8120,2025-01-03,8.0,33.0,799.0,8,0,7.5,9.5,True,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,0,0,0
3,EZ9894,2025-01-06,9.5,44.5,799.0,9,9,7.5,9.5,True,...,2024-01-06 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-06,1,1,1
4,FA8957,2025-01-03,9.5,42.5,799.0,9,8,7.5,8.5,True,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,1,1,0


In [15]:
df['prospect_renewal_date'] = pd.to_datetime(
    df['prospect_renewal_date'], errors='coerce'
)

df['datetime_out'] = pd.to_datetime(
    df['datetime_out'], errors='coerce'
)

In [ ]:
df = df.dropna(subset=['prospect_renewal_date', 'datetime_out'])
df['cutoff_date'] = df['prospect_renewal_date'] - timedelta(days=14)

In [19]:
df['prospect_renewal_date'].head()


0   2024-05-11
1   2025-09-08
2   2025-12-03
7   2024-08-11
9   2025-04-04
Name: prospect_renewal_date, dtype: datetime64[ns]

In [20]:
df = df[df['datetime_out'] <= df['cutoff_date']]

In [21]:
agg_df = df.groupby('co_ref').agg({
    'amount': ['sum', 'mean', 'count'],
    'datetime_out': ['max']
}).reset_index()

agg_df.columns = [
    'co_ref',
    'total_spent',
    'avg_payment',
    'num_payments',
    'last_payment_date'
]

In [22]:
cutoff_map = df[['co_ref', 'cutoff_date']].drop_duplicates()

agg_df = agg_df.merge(cutoff_map, on='co_ref', how='left')

agg_df['days_since_last_payment'] = (
    agg_df['cutoff_date'] - agg_df['last_payment_date']
).dt.days

In [23]:
df['days_before_cutoff'] = (df['cutoff_date'] - df['datetime_out']).dt.days

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].count().rename('payments_last_30'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 90].groupby('co_ref')['amount'].count().rename('payments_last_90'),
    on='co_ref', how='left'
)

In [24]:
agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].sum().rename('spend_last_30'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 90].groupby('co_ref')['amount'].sum().rename('spend_last_90'),
    on='co_ref', how='left'
)

In [25]:
last_30 = df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].sum()

prev_30 = df[
    (df['days_before_cutoff'] > 30) & (df['days_before_cutoff'] <= 60)
].groupby('co_ref')['amount'].sum()

trend = (last_30 - prev_30).rename('payment_trend')

agg_df = agg_df.merge(trend, on='co_ref', how='left')

In [26]:
# Tenure
agg_df = agg_df.merge(
    df[['co_ref', 'tenure_years']].drop_duplicates(),
    on='co_ref', how='left'
)

# Payment method (mode)
payment_mode = df.groupby('co_ref')['payment_method'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
)

agg_df = agg_df.merge(payment_mode.rename('payment_method_mode'), on='co_ref', how='left')

In [27]:
agg_df = agg_df.fillna(0)

In [29]:
agg_df.to_csv("../../data/03_final/final_billings_features.csv", index=False)